# Best models per category vs. SMARD

Top picks per category, their scoreboards and two plots each, read from the exports of
[`regression-models-claude.ipynb`](regression-models-claude.ipynb) (`data/models/`) and SMARD's hourly errors (`data/metrics/`); method, leakage and caveats live there.
Pick rule: per category, the top `N_PICKS` `CANDIDATE_SPLIT` rows that beat SMARD there, on average accuracy (`overall`) or in both views of the bin (the other four).

## 1 Setup and self-check

### 1.1 Settings

In [ ]:
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

N_PICKS = 3                 # top picks per category
MIN_BIN_SHARE = 0.5         # by-forecast hours / by-actual hours, tail categories
CANDIDATE_SPLIT = "rolling" # only these rows compete

# Label and colour per row, copied from MODELS and FIXED in regression-models-claude.ipynb
STYLE = {
    "sarimax_fourier": {"label": "SARIMAX + Fourier", "color": "#D9A53A"},
    "lgbm_direct": {"label": "LightGBM direct", "color": "#2C6EBA"},
    "lgbm_hybrid": {"label": "LightGBM hybrid", "color": "#2F8F5B"},
    "xgb_direct": {"label": "XGBoost direct", "color": "#E95D0F"},
    "xgb_hybrid": {"label": "XGBoost hybrid", "color": "#B10F0F"},
    "linear_direct": {"label": "Ridge direct", "color": "#7A4FA3"},
    "random_forest_hybrid": {"label": "Random forest hybrid", "color": "#8C564B"},
    "actual": {"label": "Actual residual load", "color": "#1C1C1C"},
    "smard": {"label": "SMARD day-ahead", "color": "#48505A", "linestyle": "--"},
}

# Bin regions and edges on a residual-load axis, never a model's colour
TAIL_COLOR = {
    "low_extreme": "#17BECF",   # cyan
    "below_zero": "#9EDAE5",    # light cyan
    "high_extreme": "#E6B800",  # gold
    "ordinary": "#E3E8EF",      # light grey
}


def label_of(row):
    """Display label of a (model, split_method) row."""
    return STYLE[row[0]]["label"]


def style_timeseries(ax, title, ylabel):
    """Apply the EDA time-series chart style to the matplotlib axes.

    `ylabel` is required: every plot must state its unit.
    """
    ax.set_title(title, loc="center", fontsize=15, pad=12)
    ax.set_xlabel("")
    ax.set_ylabel(ylabel, color="grey")

    # Light horizontal grid lines behind the data, no box on the top and right
    ax.grid(axis="y", color="0.9", linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)

    # Tick labels without tick marks
    ax.tick_params(colors="black", length=0)

    # x axis: one label per year, minor ticks every quarter
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_minor_locator(mdates.MonthLocator((1, 4, 7, 10)))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    # y axis: thousands separator, no decimals (e.g. 45,000)
    ax.yaxis.set_major_formatter(lambda value, _: f"{value:,.0f}")


def style_plain(ax, title, ylabel, xlabel):
    """The same look for plots without a time axis."""
    ax.set_title(title, loc="center", fontsize=15, pad=12)
    ax.set_xlabel(xlabel, color="grey")
    ax.set_ylabel(ylabel, color="grey")
    ax.grid(axis="y", color="0.9", linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.tick_params(colors="black", length=0)
    ax.yaxis.set_major_formatter(lambda value, _: f"{value:,.0f}")

### 1.2 Loading

In [ ]:
# data/: first in the working directory, then in each parent
DATA_DIR = next(
    (folder / "data" for folder in [Path.cwd(), *Path.cwd().parents] if (folder / "data").is_dir()),
    None,
)
if DATA_DIR is None:
    raise RuntimeError(f"No data/ folder found in {Path.cwd()} or any parent folder.")

MODEL_SOURCE = "notebooks/05_modeling/regression-models-claude.ipynb with EXPORT_ENABLED = True"
SMARD_SOURCE = "notebooks/02_forecast_metrics/forecast-metrics-claude.ipynb"
SOURCES = {
    "scoreboard": ("models/model_scoreboard.csv", MODEL_SOURCE),
    "hourly": ("models/model_forecast_errors_hourly.csv", MODEL_SOURCE),
    "smard": ("metrics/smard_forecast_errors_hourly.csv", SMARD_SOURCE),
}


def load(name):
    """One export as a DataFrame; stops if the file is missing."""
    path, producer = SOURCES[name]
    path = DATA_DIR / path
    if not path.exists():
        raise FileNotFoundError(f"{path} not found. Create it by running {producer}.")
    frame = pd.read_csv(path)
    if "timestamp" in frame.columns:
        frame["timestamp"] = pd.to_datetime(frame["timestamp"], format="%Y-%m-%d %H:%M:%S")
    return frame


scoreboard = load("scoreboard")
hourly = load("hourly")
smard = load("smard").set_index("timestamp")

test_start, test_end = hourly["timestamp"].min(), hourly["timestamp"].max()
print(f"test window: {test_start:%Y-%m-%d} .. {test_end:%Y-%m-%d} ({hourly['timestamp'].nunique():,} hours)")
print("rows found :")
for split, rows in scoreboard.groupby("split_method")["model"]:
    print(f"  {split:<8}: {', '.join(rows.unique())}")
print(f"SMARD file : {smard.index.min():%Y-%m-%d %H:%M} .. {smard.index.max():%Y-%m-%d %H:%M}")

### 1.3 Rebuilt quantities

Common hours, bin edges and full months, with the rules of §7.1–§7.3 of the regression notebook.

In [ ]:
SMARD_ROW = ("smard", "none")
TEST_HOURS = pd.DatetimeIndex(hourly["timestamp"].unique()).sort_values()
TEST_DAYS = TEST_HOURS.normalize().unique()
ACTUAL = hourly.groupby("timestamp")["residual_load"].first().reindex(TEST_HOURS)

# One column per (model, split_method) row
FORECAST = hourly.pivot(index="timestamp", columns=["model", "split_method"], values="forecast")
ERROR = hourly.pivot(index="timestamp", columns=["model", "split_method"], values="err_residual_load")

# Forecast per row: SMARD first, then every row with at least one forecast
ROWS = {SMARD_ROW: smard["fc_residual_load"].reindex(TEST_HOURS)}
ROWS |= {row: FORECAST[row].reindex(TEST_HOURS) for row in FORECAST if FORECAST[row].notna().any()}

# Common hours: test hours with an actual and a forecast from every row
has_all = ACTUAL.notna() & pd.concat(ROWS, axis=1).notna().all(axis=1)
COMMON = TEST_HOURS[has_all.to_numpy()]

# Error per row on the common hours (forecast − actual)
ERRORS = {SMARD_ROW: smard.loc[COMMON, "err_residual_load"]}
ERRORS |= {row: ERROR.loc[COMMON, row] for row in ROWS if row != SMARD_ROW}

# Bin edges: quantiles of the actual over the test hours
TAIL_LEVELS = [0.01, 0.25, 0.75, 0.99]
EDGES = ACTUAL.quantile(TAIL_LEVELS)
P1, P25, P75, P99 = EDGES.to_numpy()
BINS = ["low_extreme", "ordinary", "high_extreme"]
ZERO = 0.0  # MWh: below_zero bin
REPORTED_BINS = ["low_extreme", "below_zero", "ordinary", "high_extreme"]
BIN_RULE = {
    "low_extreme": f"≤ P1 ({P1:,.0f} MWh)",
    "below_zero": "< 0 MWh",
    "ordinary": f"P25..P75 ({P25:,.0f}..{P75:,.0f} MWh)",
    "high_extreme": f"> P99 ({P99:,.0f} MWh)",
}


def tail_bin(values):
    """low_extreme (<= P1), ordinary (P25..P75), high_extreme (> P99), else other."""
    conditions = [values <= P1, (values >= P25) & (values <= P75), values > P99]
    return pd.Series(np.select(conditions, BINS, default="other"), index=values.index)


def bin_masks(values):
    """One mask per reported bin; below_zero overlaps low_extreme."""
    bins = tail_bin(values)
    masks = {name: bins == name for name in BINS}
    masks["below_zero"] = values < ZERO
    return {name: masks[name] for name in REPORTED_BINS}


# Full months: first and last day inside the test window; monthly MAE on the common hours
FULL_MONTHS = pd.PeriodIndex(
    [
        month
        for month in TEST_DAYS.to_period("M").unique()
        if month.start_time in TEST_DAYS and month.end_time.normalize() in TEST_DAYS
    ]
)
month_of = COMMON.to_period("M")
MONTHLY_MAE = pd.concat(
    {row: error.abs().groupby(month_of).mean() for row, error in ERRORS.items()}, axis=1
).loc[FULL_MONTHS]

# Scoreboard: one value and one count per row and metric
VALUE = scoreboard.pivot(index=["model", "split_method"], columns="metric", values="value")
COUNT = scoreboard.pivot(index=["model", "split_method"], columns="metric", values="count")

# Candidates: registry rows of CANDIDATE_SPLIT; SMARD and seasonal naive never compete
CANDIDATES = [
    row
    for row in VALUE.index
    if row[1] == CANDIDATE_SPLIT and row[0] not in ("smard", "seasonal_naive")
]
if not CANDIDATES:
    splits = sorted({split for _, split in VALUE.index} - {"none"})
    raise ValueError(
        f"No {CANDIDATE_SPLIT!r} rows in the exports (switched off in the last run?). "
        f"Set CANDIDATE_SPLIT to one of {splits}."
    )
missing_style = sorted({model for model, _ in CANDIDATES} - STYLE.keys())
if missing_style:
    raise KeyError(
        f"No STYLE entry for {missing_style}. "
        "Copy its label and colour from MODELS in regression-models-claude.ipynb."
    )

print(f"common hours: {len(COMMON):,} of {len(TEST_HOURS):,} test hours")
print("edges       : " + ", ".join(f"P{level:.0%}".replace("%", "") + f" {edge:,.0f}" for level, edge in EDGES.items()) + " MWh")
print(f"full months : {len(FULL_MONTHS)} ({FULL_MONTHS[0]} .. {FULL_MONTHS[-1]})")
print(f"candidates  : {len(CANDIDATES)} {CANDIDATE_SPLIT} rows")

### 1.4 Self-check

In [ ]:
REGENERATE_MODELS = f"data/models/ ({MODEL_SOURCE})"
REGENERATE_ALL = f"data/metrics/ ({SMARD_SOURCE}), then {REGENERATE_MODELS}"


def require(ok, problem, regenerate):
    """Stop the notebook, naming the files to regenerate."""
    if not ok:
        raise RuntimeError(f"self-check failed: {problem}. Regenerate {regenerate}.")


# Same actuals in the model export and the SMARD file
actual_gap = float((ACTUAL[COMMON] - smard.loc[COMMON, "residual_load"]).abs().max())
require(
    actual_gap <= 1e-6,
    f"residual_load differs between the two hourly files by up to {actual_gap:,.2f} MWh",
    REGENERATE_ALL,
)

# Same rows in the scoreboard and the hourly export
row_gap = sorted(set(VALUE.index) ^ set(ERRORS))
require(not row_gap, f"rows {row_gap} are in only one of the two model exports", REGENERATE_MODELS)

# MAE and hour count per row
for row, error in ERRORS.items():
    mae, hours = error.abs().mean(), len(error)
    exported_mae, exported_hours = VALUE.loc[row, "MAE"], COUNT.loc[row, "MAE"]
    require(
        abs(mae - exported_mae) <= 1e-6 and hours == exported_hours,
        f"{row}: MAE {mae:,.3f} on {hours:,} h, scoreboard {exported_mae:,.3f} on {exported_hours:,.0f} h",
        REGENERATE_ALL if row == SMARD_ROW else REGENERATE_MODELS,
    )

# By-actual hours per bin
for name, in_bin in bin_masks(ACTUAL[COMMON]).items():
    exported = COUNT[f"MAE_{name}_by_actual"]
    require(
        (exported == in_bin.sum()).all(),
        f"{name}: {in_bin.sum():,} by-actual hours, scoreboard {sorted(int(count) for count in exported.unique())}",
        REGENERATE_MODELS,
    )

# Months beating SMARD
for row in ERRORS:
    if row == SMARD_ROW:
        continue
    months = int((MONTHLY_MAE[row] < MONTHLY_MAE[SMARD_ROW]).sum())
    exported = (VALUE.loc[row, "months_beating_smard"], COUNT.loc[row, "months_beating_smard"])
    require(
        (months, len(FULL_MONTHS)) == exported,
        f"{row}: {months} / {len(FULL_MONTHS)} months beating SMARD, scoreboard {exported[0]:.0f} / {exported[1]:.0f}",
        REGENERATE_MODELS,
    )

print(
    f"self-check passed: exports reproduce the scoreboard "
    f"({len(ERRORS)} rows, {len(COMMON):,} common hours)"
)